In [1]:
from google.colab import files
files.upload()   # click "Choose Files" and pick dist/nuskha-dwbi-colab.zip from the project folder
!unzip -oq /content/nuskha-dwbi-colab.zip -d /content/nuskha-dwbi
%cd /content/nuskha-dwbi
import os; os.environ['NUSKHA_MODE'] = 'FULL_REBUILD'

Saving nuskha-dwbi-colab.zip to nuskha-dwbi-colab.zip
/content/nuskha-dwbi


# Nuskha Pharmacies data warehouse: full pipeline

CT-472 Data Warehousing and BI project. **All data is synthetic (seed 472); no real patients, brands or companies.**

Each cell below calls code in `src/`; the notebook holds no logic of its own. Steps print `[OK]`, `[SKIP]` or `[FAILED]` with the last lines of their output.

## Setup
In Google Colab this installs and starts MySQL and PostgreSQL and creates the users (`scripts/setup_env.sh colab`). On a local machine the database services already run, so it only checks the five connections.

In [2]:
import os, subprocess, sys
IN_COLAB = 'COLAB_RELEASE_TAG' in os.environ or 'COLAB_GPU' in os.environ
if IN_COLAB:
    if not os.path.exists('src'):
        raise SystemExit('Upload or clone the project folder and %cd into it first.')
    subprocess.run(['bash', 'scripts/setup_env.sh', 'colab'], check=True)
subprocess.run([sys.executable, 'scripts/check_env.py', '--create'], check=True)

CompletedProcess(args=['/usr/bin/python3', 'scripts/check_env.py', '--create'], returncode=0)

## Mode
`FULL_REBUILD` empties the warehouse and runs everything from the sources; `CONTINUE` keeps what exists and only adds what is missing (safe to repeat). The headless test run sets it with the environment variable `NUSKHA_MODE`.

In [3]:
import os
MODE = os.environ.get('NUSKHA_MODE', 'CONTINUE')   # or 'FULL_REBUILD'
from src import pipeline
if MODE == 'FULL_REBUILD':
    pipeline.reset()
print('mode:', MODE)

warehouse schemas dropped; raw snapshots, snapshots and stage logs removed
mode: FULL_REBUILD


## 1. Source systems
Four synthetic department databases are generated from seed 472 (FULL_REBUILD only) and loaded into MySQL (Sales, Finance), PostgreSQL (Purchasing) and SQLite (Inventory). The load is done twice and the per-table checksums are compared.

**Generator: reference data + stage-1 sources (Nov 2023 - Aug 2026)** — `python -m src.generate_sources --stage 1`

In [4]:
pipeline.run_step('generate_stage_1', MODE)

[OK] generate_stage_1: Generator: reference data + stage-1 sources (Nov 2023 - Aug 2026) (21.7 s)
         1 purchasing_db            branch    10
         1 purchasing_db       vendor_item   152
         1 purchasing_db    purchase_order  1726
         1 purchasing_db     goods_receipt  1717
         1 purchasing_db          grn_line  4683
         1     inventory              item   150
         1     inventory             store    10
         1     inventory       stock_count 16500
         1     inventory item_cost_history   600
         1    finance_db     panel_account    15
         1    finance_db           fx_rate  1035
         1    finance_db     panel_invoice   412


{'step': 'generate_stage_1', 'status': 'OK', 'seconds': 21.7}

**Generator: stage-2 sources (Sep - Oct 2026, corrections, deletions)** — `python -m src.generate_sources --stage 2`

In [5]:
pipeline.run_step('generate_stage_2', MODE)

[OK] generate_stage_2: Generator: stage-2 sources (Sep - Oct 2026, corrections, deletions) (19.8 s)
    /content/nuskha-dwbi/src/gen/sources.py:172: SettingWithCopyWarning: 
    A value is trying to be set on a copy of a slice from a DataFrame.
    Try using .loc[row_indexer,col_indexer] = value instead
    
    See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
      ln["updated_at"] = mysql_ts(ln.updated_at)
    /content/nuskha-dwbi/src/gen/sources.py:174: SettingWithCopyWarning: 
    A value is trying to be set on a copy of a slice from a DataFrame.
    Try using .loc[row_indexer,col_indexer] = value instead
    
    See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
      ln[c] = money_text(ln[c])


{'step': 'generate_stage_2', 'status': 'OK', 'seconds': 19.8}

**Warehouse DDL: schemas dw, stg, etl, meta, ai** — `python -m src.create_warehouse`

In [6]:
pipeline.run_step('create_warehouse', MODE)

[OK] create_warehouse: Warehouse DDL: schemas dw, stg, etl, meta, ai (0.9 s)
    01_schemas_and_control.sql: done
    02_dw_stars.sql: done
    03_meta_and_ai.sql: done
    41 statements run


{'step': 'create_warehouse', 'status': 'OK', 'seconds': 0.9}

**Load stage 1 into MySQL, PostgreSQL, SQLite (twice, checksums compared)** — `python -m src.load_sources --stage 1 --twice`

In [7]:
pipeline.run_step('load_sources_stage_1', MODE)

[OK] load_sources_stage_1: Load stage 1 into MySQL, PostgreSQL, SQLite (twice, checksums compared) (15.4 s)
    purchasing_db            branch           10 977a367412d44e97f95461b17679c74c           10 977a367412d44e97f95461b17679c74c       True   PASS
    purchasing_db       vendor_item          152 ddb39754db85689d419cc2501e58cc70          152 ddb39754db85689d419cc2501e58cc70       True   PASS
    purchasing_db    purchase_order         1726 54ff83539eb89f6941179e5338f4fe66         1726 54ff83539eb89f6941179e5338f4fe66       True   PASS
    purchasing_db     goods_receipt         1717 7f902e72fd20ae9249eaa068b9296ca7         1717 7f902e72fd20ae9249eaa068b9296ca7       True   PASS
    purchasing_db          grn_line         4683 5e779706dba4059f35bc42a7d963d3a9         4683 5e779706dba4059f35bc42a7d963d3a9       True   PASS
        inventory              item          150 e141e6192c1a993e40d0dedb714e9e43          150 e141e6192c1a993e40d0dedb714e9e43       True   PASS
        inventor

{'step': 'load_sources_stage_1', 'status': 'OK', 'seconds': 15.4}

## 2. Batch 1: extract, clean, match
Transaction tables are extracted by `updated_at` watermark into an untouched raw snapshot. Staging normalizes dates, units, money and currency; crosswalks give one product, branch and customer identity; bad rows go to `etl.rejects` with a reason code; rows and amounts are reconciled.

**Extract batch 1 by watermark into a raw snapshot** — `python -m src.extract --to 2026-08-31 23:59:59 --double-export data/generated/stage1/double_export.csv`

In [8]:
pipeline.run_step('extract_batch_1', MODE)

[OK] extract_batch_1: Extract batch 1 by watermark into a raw snapshot (3.1 s)
            1 purchasing_db            branch              10            10               NaN            NaN                10          True           True
            1 purchasing_db       vendor_item             152           152               NaN            NaN               152          True           True
            1 purchasing_db    purchase_order            1726          1726               NaN            NaN              1726          True           True
            1 purchasing_db     goods_receipt            1717          1717               NaN            NaN              1717          True           True
            1 purchasing_db          grn_line            4683          4683       12741839.81    12741839.81              4706          True           True
            1     inventory              item             150           150               NaN            NaN               150          True 

{'step': 'extract_batch_1', 'status': 'OK', 'seconds': 3.1}

**ETL batch 1: profile before** — `python -m src.etl_db profile-before --batch 1`

In [9]:
pipeline.run_step('etl_profile-before_1', MODE)

[OK] etl_profile-before_1: ETL batch 1: profile before (3.2 s)
    check  expected  actual result
    profile_before covers every source table        17      17   PASS
    
    1 PASS, 0 FAIL


{'step': 'etl_profile-before_1', 'status': 'OK', 'seconds': 3.2}

**ETL batch 1: stage** — `python -m src.etl_db stage --batch 1`

In [10]:
pipeline.run_step('etl_stage_1', MODE)

[OK] etl_stage_1: ETL batch 1: stage (16.5 s)
    check  expected  actual result
                                                stg tables written        17      17   PASS
                                       no text date columns in stg         0       0   PASS
           no float (double/real) columns in stg: money is NUMERIC         0       0   PASS
    Inventory categories all canonical (8 names or 'Not recorded')         0       0   PASS
    
    4 PASS, 0 FAIL


{'step': 'etl_stage_1', 'status': 'OK', 'seconds': 16.5}

**ETL batch 1: profile after** — `python -m src.etl_db profile-after --batch 1`

In [11]:
pipeline.run_step('etl_profile-after_1', MODE)

[OK] etl_profile-after_1: ETL batch 1: profile after (6.4 s)
    check  expected  actual result
    profile_before_after has a before/after pair per table        34      34   PASS
    
    1 PASS, 0 FAIL


{'step': 'etl_profile-after_1', 'status': 'OK', 'seconds': 6.4}

**ETL batch 1: crosswalk** — `python -m src.etl_db crosswalk --batch 1`

In [12]:
pipeline.run_step('etl_crosswalk_1', MODE)

[OK] etl_crosswalk_1: ETL batch 1: crosswalk (3.0 s)
                          crosswalk: xw_branch: sales_db keys matched by 'rule'     info       9   PASS
             crosswalk: xw_branch: sales_db keys matched by 'rule (alias list)'     info       9   PASS
                     crosswalk: xw_branch: sales_db keys matched by 'unmatched'     info       6   PASS
    crosswalk: xw_customer: source keys mapping to more than one canonical code        0       0   PASS
      crosswalk: xw_customer: keys with neither a code nor the 'unmatched' flag        0       0   PASS
                     crosswalk: xw_customer: finance_db keys matched by 'exact'     info      15   PASS
                       crosswalk: xw_customer: sales_db keys matched by 'exact'     info    1215   PASS
                        crosswalk: xw_customer: sales_db keys matched by 'rule'     info      15   PASS
           crosswalk: xw_customer: sales_db keys matched by 'rule (alias list)'     info       9   PASS
           

{'step': 'etl_crosswalk_1', 'status': 'OK', 'seconds': 3.0}

**ETL batch 1: prejoin** — `python -m src.etl_db prejoin --batch 1`

In [13]:
pipeline.run_step('etl_prejoin_1', MODE)

[OK] etl_prejoin_1: ETL batch 1: prejoin (2.4 s)
             join 'receipt_line -> xw_product' keeps the row count     28099   28099   PASS
                   join 'receipt -> xw_branch' keeps the row count     28099   28099   PASS
          join 'receipt -> xw_customer (card)' keeps the row count     28099   28099   PASS
    join 'receipt -> xw_customer (panel text)' keeps the row count     28099   28099   PASS
              join 'grn_line -> goods_receipt' keeps the row count      4683    4683   PASS
        join 'goods_receipt -> purchase_order' keeps the row count      4683    4683   PASS
                join 'grn_line -> vendor_item' keeps the row count      4683    4683   PASS
                 join 'grn_line -> xw_product' keeps the row count      4683    4683   PASS
            join 'purchase_order -> xw_branch' keeps the row count      4683    4683   PASS
             join 'purchase_order -> supplier' keeps the row count      4683    4683   PASS
    
    26 PASS, 0 FAIL


{'step': 'etl_prejoin_1', 'status': 'OK', 'seconds': 2.4}

**ETL batch 1: quality** — `python -m src.etl_db quality --batch 1`

In [14]:
pipeline.run_step('etl_quality_1', MODE)

[OK] etl_quality_1: ETL batch 1: quality (2.6 s)
         receipt_line R05_UNKNOWN_BRANCH: ETL result = planted ledger (key for key)        56      56   PASS
            receipt_line R07_ORPHAN_LINE: ETL result = planted ledger (key for key)        56      56   PASS
    receipt_line U01_UNRESOLVED_CUSTOMER: ETL result = planted ledger (key for key)        20      20   PASS
            grn_line D01_EXACT_DUPLICATE: ETL result = planted ledger (key for key)        23      23   PASS
            grn_line R01_MISSING_PRODUCT: ETL result = planted ledger (key for key)        28      28   PASS
                    grn_line R02_BAD_QTY: ETL result = planted ledger (key for key)        19      19   PASS
                   grn_line R03_BAD_DATE: ETL result = planted ledger (key for key)        14      14   PASS
                 grn_line R04_BAD_AMOUNT: ETL result = planted ledger (key for key)        14      14   PASS
                 grn_line R06_NO_FX_RATE: ETL result = planted ledger (key for 

{'step': 'etl_quality_1', 'status': 'OK', 'seconds': 2.6}

**ETL batch 1: reconcile** — `python -m src.etl_db reconcile --batch 1`

In [15]:
pipeline.run_step('etl_reconcile_1', MODE)

[OK] etl_reconcile_1: ETL batch 1: reconcile (2.8 s)
    batch_id                  table  source_rows  staged_valid  rejected  duplicates_removed  balanced
            1  sales_db.receipt_line        28238         27543       556                 139      True
            1 purchasing_db.grn_line         4706          4604        79                  23      True
                                                                    check    expected      actual result
         receipt_line: source rows = staged valid + rejected + duplicates    28238.00    28238.00   PASS
    receipt_line: amount per month source = valid + rejected + duplicates        0.00        0.00   PASS
                  receipt_line: staged amount total = raw snapshot amount 15281835.85 15281835.85   PASS
             grn_line: source rows = staged valid + rejected + duplicates     4706.00     4706.00   PASS
        grn_line: amount per month source = valid + rejected + duplicates        0.00        0.00   PASS
      

{'step': 'etl_reconcile_1', 'status': 'OK', 'seconds': 2.8}

## 3. Batch 1: warehouse load
Dimensions, then facts, as upserts on natural keys in one transaction per fact table, with lineage.

**Load batch 1 into the stars (dimensions, facts, lineage)** — `python -m src.load_db --batch 1`

In [16]:
pipeline.run_step('load_batch_1', MODE)

[OK] load_batch_1: Load batch 1 into the stars (dimensions, facts, lineage) (22.5 s)
    batch_id            table  rows_sent  inserted  updated
            1      dw.dim_date       1188      1188        0
            1   dw.dim_product        151       151        0
            1    dw.dim_branch         11        11        0
            1  dw.dim_customer       1217      1217        0
            1  dw.dim_supplier         26        26        0
            1    dw.fact_sales      27543     27543        0
            1 dw.fact_purchase       4604      4604        0


{'step': 'load_batch_1', 'status': 'OK', 'seconds': 22.5}

**Dimension checks** — `python -m src.dw_checks dims --batch 1`

In [17]:
pipeline.run_step('check_dims', MODE)

[OK] check_dims: Dimension checks (1.1 s)
              dim_customer: special members [-1, 0] present                                   2                                   2   PASS
            dim_customer: business key customer_code unique                                   0                                   0   PASS
                        dim_customer: no NULL business keys                                   0                                   0   PASS
                 dim_supplier: special members [-1] present                                   1                                   1   PASS
            dim_supplier: business key supplier_code unique                                   0                                   0   PASS
                        dim_supplier: no NULL business keys                                   0                                   0   PASS
        dim_date: 2023-11-01 to 2027-01-31, one row per day 1188 rows, 2023-11-01 to 2027-01-31 1188 rows, 2023-11-01 to 202

{'step': 'check_dims', 'status': 'OK', 'seconds': 1.1}

**Fact checks (rows, amounts, Diabetes Care margin)** — `python -m src.dw_checks facts --batch 1`

In [18]:
pipeline.run_step('check_facts', MODE)

[OK] check_facts: Fact checks (rows, amounts, Diabetes Care margin) (2.7 s)
                                         fact_sales: rows = valid staged rows          27543    27543.000   PASS
                                               fact_sales: natural key unique              0        0.000   PASS
                                      fact_sales: every row has a lineage row              0        0.000   PASS
                                      fact_purchase: rows = valid staged rows           4604     4604.000   PASS
                                            fact_purchase: natural key unique              0        0.000   PASS
                                   fact_purchase: every row has a lineage row              0        0.000   PASS
              fact_sales: net amount = valid staged net amount (to the paisa)    15070308.45 15070308.450   PASS
                           fact_sales: gross margin = net - cost on every row              0        0.000   PASS
                    

{'step': 'check_facts', 'status': 'OK', 'seconds': 2.7}

## 4. Batches, SCD Type 1 and failure handling
An unchanged rerun changes nothing; batch 2 brings late receipts, corrections, cancellations and deletions; the cough-syrup category fix is an SCD Type 1 overwrite; a forced failure rolls back.

**Rerun batch 1 unchanged: 0 rows change** — `python -m src.batch_tests rerun`

In [19]:
pipeline.run_step('rerun_test', MODE)

[OK] rerun_test: Rerun batch 1 unchanged: 0 rows change (24.6 s)
            1      dw.dim_date       1188         0        0
            1   dw.dim_product        151         0        0
            1    dw.dim_branch         11         0        0
            1  dw.dim_customer       1217         0        0
            1  dw.dim_supplier         26         0        0
            1    dw.fact_sales      27543         0        0
            1 dw.fact_purchase       4604         0        0
                                                        check  expected  actual result
                 rerun of batch 1: 0 rows written by the load         0       0   PASS
    rerun of batch 1: totals and table fingerprints unchanged         0       0   PASS
    
    2 PASS, 0 FAIL


{'step': 'rerun_test', 'status': 'OK', 'seconds': 24.6}

**Stage-2 sources, extract batch 2, ETL, load, deletion check** — `python -m src.batch_tests batch2`

In [20]:
pipeline.run_step('batch_2', MODE)

[OK] batch_2: Stage-2 sources, extract batch 2, ETL, load, deletion check (33.1 s)
                                              deleted receipts' rows kept in fact_sales (none removed)                                                 > 0                                                   6   PASS
                                               batch 2 extracted by watermark (from = batch 1 cut-off)                                 2026-08-31 23:59:59                                 2026-08-31 23:59:59   PASS
                                             late receipts loaded into their real month (Jul-Aug 2026)                                         40 receipts                       40 receipts in 202607, 202608   PASS
                    stage-1 lines with a value change updated in place (qty fixes, panel fixes, voids)                                                  71                                                  71   PASS
                                    re-saved lines with no va

{'step': 'batch_2', 'status': 'OK', 'seconds': 33.1}

**SCD Type 1 category totals before / after** — `python -m src.batch_tests scd1`

In [21]:
pipeline.run_step('scd1', MODE)

[OK] scd1: SCD Type 1 category totals before / after (1.3 s)
    check                 expected                   actual result
     SCD1: exactly two category totals move (Gastro, Cough and Cold) [Cough and Cold, Gastro] [Cough and Cold, Gastro]   PASS
                                       SCD1: company total unchanged                      0.0                      0.0   PASS
    SCD1: one product changed category (overwritten, no history row)                        1                        1   PASS
            SCD1: stored margins of untouched batch-1 rows unchanged                        0                        0   PASS
    
    4 PASS, 0 FAIL


{'step': 'scd1', 'status': 'OK', 'seconds': 1.3}

**FAIL_AT=0.5 rollback and clean rerun** — `python -m src.batch_tests failure`

In [22]:
pipeline.run_step('controlled_failure', MODE)

[OK] controlled_failure: FAIL_AT=0.5 rollback and clean rerun (10.5 s)
            4  dw.dim_customer       1217         0        0
            4  dw.dim_supplier         26         0        0
            4    dw.fact_sales       2369         0        0
            4 dw.fact_purchase        321         0        0
                                                            check expected                                                       actual result
          FAIL_AT=0.5 raised an error halfway through a fact load    error FAIL_AT=0.5: controlled failure after 1184 of 2369 rows of f   PASS
                     failed batch recorded as FAILED in etl.batch   FAILED                                                       FAILED   PASS
    rollback: totals and fingerprints unchanged after the failure        0                                                            0   PASS
                    rerun after the failure matches the clean run        0                                       

{'step': 'controlled_failure', 'status': 'OK', 'seconds': 10.5}

## 5. Metadata, lineage and traces
The metadata repository, ERDs reverse-engineered from each engine's catalog, and six traces from a warehouse figure back to a source row.

**Metadata repository and lineage coverage** — `python -m src.metadata --load`

In [23]:
pipeline.run_step('metadata', MODE)

[OK] metadata: Metadata repository and lineage coverage (1.2 s)
    check expected  actual result
               warehouse columns with at least one mapping       67    67.0   PASS
          mappings to columns that do not exist in the DDL        0     0.0   PASS
                 mappings naming an unknown transform rule        0     0.0   PASS
                                   meta.source_system rows      > 0     4.0   PASS
                                   meta.source_object rows      > 0    34.0   PASS
                                  meta.transform_rule rows      > 0    19.0   PASS
                                      meta.column_map rows      > 0    70.0   PASS
                                         meta.lineage rows      > 0 34738.0   PASS
                     dw.fact_sales: % of rows with lineage      100   100.0   PASS
                  dw.fact_purchase: % of rows with lineage      100   100.0   PASS
    meta.column_map rules all exist in meta.transform_rule        0     

{'step': 'metadata', 'status': 'OK', 'seconds': 1.2}

**Reverse engineering: MySQL, PostgreSQL, SQLite** — `python -m src.reverse_engineer`

In [24]:
pipeline.run_step('reverse_engineer', MODE)

[OK] reverse_engineer: Reverse engineering: MySQL, PostgreSQL, SQLite (2.7 s)
    department                   engine  tables  observed_links  inferred_links
         sales           MySQL sales_db       4               0               2
    purchasing PostgreSQL purchasing_db       6               6               0
     inventory      SQLite inventory.db       4               0               3


{'step': 'reverse_engineer', 'status': 'OK', 'seconds': 2.7}

## 6. Analytics and AI
Six saved views, twelve analytic queries, the sales forecast and anomaly detection (run twice to show the retry adds no rows).

**Saved views and their total checks** — `python -m src.views`

In [25]:
pipeline.run_step('views', MODE)

[OK] views: Saved views and their total checks (1.5 s)
    6 views created: 01_vw_sales_enriched, 02_vw_sales_product_month, 03_vw_purchase_product_month, 04_vw_margin_bridge, 05_vw_forecast_features, 06_vw_region_category_month
                                                                           check    expected      actual result
                          vw_sales_enriched net = fact_sales net (not cancelled) 16382871.40 16382871.40   PASS
                                     vw_sales_product_month net = fact_sales net 16382871.40 16382871.40   PASS
                               vw_sales_product_month margin = fact_sales margin  4097328.36  4097328.36   PASS
               vw_purchase_product_month PKR = fact_purchase PKR (not cancelled) 16546048.98 16546048.98   PASS
                                       vw_forecast_features net = fact_sales net 16382871.40 16382871.40   PASS
                                   vw_region_category_month net = fact_sales net 16382871.40 163828

{'step': 'views', 'status': 'OK', 'seconds': 1.5}

**Six traces to a source row** — `python -m src.traces`

In [26]:
pipeline.run_step('traces', MODE)

[OK] traces: Six traces to a source row (4.1 s)
    trace                                                title  steps  ends_at_source_row result
       T1       A normal till receipt line (CLF-2026-000417|1)      6                True   PASS
       T2               A USD purchase line (GRN-2026-00469|1)      7                True   PASS
       T3   A late receipt loaded in batch 2 (CLF-2026-000308)      6                True   PASS
       T4 Derived measure gross_margin_pkr (CLF-2026-000109|2)      5                True   PASS
       T5               Dashboard total: FY2026 net sales card      5                True   PASS
       T6    AI input series: vw_forecast_features, March 2026      4                True   PASS


{'step': 'traces', 'status': 'OK', 'seconds': 4.1}

**Analytic queries A1-A12 with charts and readings** — `python -m src.queries`

In [27]:
pipeline.run_step('queries', MODE)

[OK] queries: Analytic queries A1-A12 with charts and readings (8.5 s)
      A01 A01_branch_quarter_comparison.sql    10   PASS
      A02     A02_category_margin_share.sql    32   PASS
      A03    A03_monthly_category_trend.sql   288   PASS
      A04           A04_asp_by_category.sql   288   PASS
      A05 A05_local_vs_import_purchases.sql    40   PASS
      A06         A06_ytd_running_sales.sql    36   PASS
      A07           A07_rolling_3_month.sql    36   PASS
      A08            A08_growth_mom_yoy.sql    36   PASS
      A09         A09_top10_rank_change.sql    10   PASS
      A10    A10_price_vs_purchase_cost.sql  3049   PASS
      A11                A11_volatility.sql     8   PASS
      A12    A12_diabetes_margin_bridge.sql    36   PASS


{'step': 'queries', 'status': 'OK', 'seconds': 8.5}

**Forecast and anomaly detection, run twice (retry)** — `python -m src.ai_run --retry`

In [28]:
pipeline.run_step('ai', MODE)

[OK] ai: Forecast and anomaly detection, run twice (retry) (14.1 s)
    method  holdout_mae_pkr  holdout_rmse_pkr  usable_training_rows
               ridge         70806.15          71562.79                    21
      seasonal_naive        116207.40         128332.85                    21
    last_month_naive         56347.42          57749.83                    21
              method  flags  true_positives  false_alerts  missed  precision  recall
    isolation_forest     22              20             2       5      0.909    0.80
            robust_z     10               6             4      19      0.600    0.24
      run  model_run  forecast_result  anomaly_result  unchanged_after_retry
    first          3                6            1152                   True
    retry          3                6            1152                   True


{'step': 'ai', 'status': 'OK', 'seconds': 14.1}

## 7. Validation and Power BI package
The 11-check validation suite, then the CSV exports, semantic model and report for Power BI.

**Validation suite (11 checks)** — `python -m src.validation`

In [29]:
pipeline.run_step('validation', MODE)

[OK] validation: Validation suite (11 checks) (2.4 s)
      6               amounts: MySQL valid lines (+ deleted receipts kept) = fact_sales, per month   0 of 36 months differ         0 months differ   PASS
      6         amounts: PostgreSQL valid GRN lines = fact_purchase (original currency), per month   0 of 36 months differ         0 months differ   PASS
      7                    join safety: dw.fact_sales total before = after joining every dimension             16400628.65             16400628.65   PASS
      7                 join safety: dw.fact_purchase total before = after joining every dimension             16567538.48             16567538.48   PASS
      8                         lineage: % of dw.fact_sales rows with source system, key and batch                     100                   100.0   PASS
      8                      lineage: % of dw.fact_purchase rows with source system, key and batch                     100                   100.0   PASS
      9               

{'step': 'validation', 'status': 'OK', 'seconds': 2.4}

**Power BI CSVs and reconciliation** — `python -m src.powerbi_export`

In [30]:
pipeline.run_step('powerbi_export', MODE)

[OK] powerbi_export: Power BI CSVs and reconciliation (7.4 s)
    total  sql_total  csv_total power_bi_total result
                            FY2026 net sales (PKR) 6230294.70 6230294.70                  PASS
    Gulshan-e-Iqbal net sales, calendar 2025 (PKR)  745398.10  745398.10                  PASS
                      October 2026 purchases (PKR)  599167.14  599167.14                  PASS


{'step': 'powerbi_export', 'status': 'OK', 'seconds': 7.4}

**Power BI semantic model (TMDL)** — `python -m src.powerbi_model`

In [31]:
pipeline.run_step('powerbi_model', MODE)

[OK] powerbi_model: Power BI semantic model (TMDL) (0.9 s)
    check  expected  actual result
          TMDL files indent with tabs only         0       0   PASS
        ratio measures use DIVIDE (no '/')         0       0   PASS
    relationship columns exist in the CSVs         0       0   PASS
         measure columns exist in the CSVs         0       0   PASS
                            tables written        10      10   PASS


{'step': 'powerbi_model', 'status': 'OK', 'seconds': 0.9}

**Power BI report (PBIR), schema-validated** — `python -m src.powerbi_report --validate`

In [32]:
pipeline.run_step('powerbi_report', MODE)

[OK] powerbi_report: Power BI report (PBIR), schema-validated (4.7 s)
          definition/pages/diagnostic/visuals/dg_bridge/visual.json definition/visualContainer/2.4.0/schema.json       0               PASS
             definition/pages/diagnostic/visuals/dg_top/visual.json definition/visualContainer/2.4.0/schema.json       0               PASS
                                      definition/pages/ai/page.json            definition/page/2.0.0/schema.json       0               PASS
                definition/pages/ai/visuals/ai_slicer_0/visual.json definition/visualContainer/2.4.0/schema.json       0               PASS
                definition/pages/ai/visuals/ai_slicer_1/visual.json definition/visualContainer/2.4.0/schema.json       0               PASS
              definition/pages/ai/visuals/ai_last_batch/visual.json definition/visualContainer/2.4.0/schema.json       0               PASS
                  definition/pages/ai/visuals/ai_footer/visual.json definition/visualConta

{'step': 'powerbi_report', 'status': 'OK', 'seconds': 4.7}

## Final warehouse totals
These must be identical between runs and between this machine and Colab.

In [33]:
t = pipeline.totals()
out = os.environ.get('NUSKHA_TOTALS_FILE')
if out:
    t.to_csv(out, index=False, lineterminator='\n')
t

,measure,value
0,fact_sales rows,29820
1,fact_sales net PKR,16400628.65
2,fact_sales gross margin PKR,4101635.58
3,fact_purchase rows,4918
4,fact_purchase PKR,16567538.48
5,etl.rejects rows,687
6,etl.deletion_log rows,3
7,meta.lineage rows,34738
8,FY2026 net sales PKR (view),6230294.70
9,ai.forecast_result rows,6
